# PDHG Tuning In Colab

This notebook keeps Colab as a thin launcher around the repo's existing Hydra scripts.

It will:
- mount Google Drive
- fetch the repo into `/content`
- install dependencies
- download the FFHQ checkpoint if needed
- create a Colab-specific PDHG tuning config
- launch staged tuning runs
- summarize the resulting leaderboard


## 1. Runtime

In Colab, go to `Runtime > Change runtime type` and choose:
- `Python 3`
- `GPU`

If Colab lets you choose a premium GPU, prefer `L4`. `A100` is even faster if available.

In [ ]:
#@title Project Settings

# Choose how the repo is fetched into Colab.
# - "git": clone from GitHub
# - "drive_zip": unzip a repo archive stored in Google Drive
SETUP_MODE = "git"  #@param ["git", "drive_zip"]

# If SETUP_MODE == "git", fill these in.
REPO_URL = "https://github.com/<your-user>/<your-repo>.git"  #@param {type:"string"}
REPO_BRANCH = "main"  #@param {type:"string"}

# If SETUP_MODE == "drive_zip", point this at a .zip in Drive.
DRIVE_ZIP_PATH = "/content/drive/MyDrive/mycode2.zip"  #@param {type:"string"}

# Local working directory inside the Colab VM.
REPO_DIR = "/content/mycode2"

# Where run artifacts should be copied back into Google Drive.
DRIVE_EXPORT_DIR = "/content/drive/MyDrive/pdhg_tuning_exports"  #@param {type:"string"}

# The stage to launch from the tuning config.
RUN_STAGE = "stage_a_steps"  #@param ["stage_a_steps", "stage_b_schedule", "stage_c_refine"]

# Optional cap on the number of runs in the selected stage.
# Set to 0 to disable the cap.
MAX_RUNS = 0  #@param {type:"integer"}

CUSTOM_TUNING_CONFIG = "tuning/pdhg_tuning.colab.yaml"


In [ ]:
#@title Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')


In [ ]:
#@title Fetch The Repo
import os
import shutil
import zipfile
from pathlib import Path

repo_dir = Path(REPO_DIR)
if repo_dir.exists():
    shutil.rmtree(repo_dir)

if SETUP_MODE == "git":
    if "<your-user>" in REPO_URL or "<your-repo>" in REPO_URL:
        raise ValueError("Fill in REPO_URL before running this cell.")
    !git clone --depth 1 --branch "$REPO_BRANCH" "$REPO_URL" "$REPO_DIR"
elif SETUP_MODE == "drive_zip":
    zip_path = Path(DRIVE_ZIP_PATH)
    if not zip_path.exists():
        raise FileNotFoundError(f"Zip file not found: {zip_path}")
    with zipfile.ZipFile(zip_path, "r") as handle:
        handle.extractall("/content")
    extracted_dirs = [path for path in Path("/content").iterdir() if path.is_dir() and path.name != "drive"]
    if repo_dir.exists():
        pass
else:
    raise ValueError(f"Unsupported SETUP_MODE: {SETUP_MODE}")

if not repo_dir.exists():
    raise FileNotFoundError(f"Repo directory was not created at {repo_dir}")

%cd /content/mycode2
!git status --short


In [ ]:
#@title Install Python Dependencies
%cd /content/mycode2
!python -m pip install --upgrade pip
!pip install -r requirements.txt


In [ ]:
#@title Download The FFHQ Checkpoint If Needed
%cd /content/mycode2
!mkdir -p pretrained-models
!test -f pretrained-models/ffhq_10m.pt || gdown --id 1BGwhRWUoguF-D8wlZ65tf227gp3cDUDh -O pretrained-models/ffhq_10m.pt
!ls -lh pretrained-models


## 2. Edit The Search Space

Replace the placeholder lists below with the numeric values you want to sweep.

This cell creates a new config at `tuning/pdhg_tuning.colab.yaml` so the template file stays untouched.

In [ ]:
#@title Build A Colab-Specific Tuning Config
from pathlib import Path
import yaml

repo_dir = Path(REPO_DIR)
template_path = repo_dir / "tuning" / "pdhg_tuning.template.yaml"
custom_path = repo_dir / CUSTOM_TUNING_CONFIG

with template_path.open("r", encoding="utf-8") as handle:
    cfg = yaml.safe_load(handle)

# Keep the algorithmic baseline aligned with test2.sh while
# leaving artifact-heavy options disabled for lighter sweeps.
cfg["study"]["name"] = "pdhg_phase_retrieval_colab"
cfg["base_overrides"] = [
    "sampler=edm_pdhg",
    "inverse_task=phase_retrieval",
    "name=PDHG_Tuning",
    "wandb=false",
    "save_samples=false",
    "save_traj=false",
    "save_traj_raw_data=false",
    "show_config=false",
    "total_images=10",
    "batch_size=10",
    "num_runs=1",
    "inverse_task.operator.sigma=0.05",
    "sampler.annealing_scheduler_config.num_steps=400",
    "inverse_task.admm_config.denoise.final_step=tweedie",
    "inverse_task.admm_config.max_iter=400",
    "inverse_task.admm_config.dys.gamma=0.0075",
    "inverse_task.admm_config.dys.lambda_schedule.activate=true",
    "inverse_task.admm_config.dys.lambda_schedule.start=1",
    "inverse_task.admm_config.dys.lambda_schedule.end=1",
    "inverse_task.admm_config.dys.lambda_schedule.warmup=0",
    "inverse_task.admm_config.denoise.lgvd.num_steps=0",
]

# Default ranges below follow the current tuning plan and keep the
# algorithmic baseline aligned with test2.sh.
stage_values = {
    "stage_a_steps": {
        "inverse_task.admm_config.pdhg.tau": [0.005, 0.0075, 0.01],
        "inverse_task.admm_config.pdhg.sigma_dual": [800, 1200, 1600],
    },
    "stage_b_schedule": {
        "sampler.annealing_scheduler_config.sigma_max": [20, 27, 30],
        "sampler.annealing_scheduler_config.sigma_min": [0.05, 0.075, 0.1],
    },
    "stage_c_refine": {
        "sampler.annealing_scheduler_config.sigma_max": [27],
        "sampler.annealing_scheduler_config.sigma_min": [0.075],
        "inverse_task.admm_config.pdhg.tau": [0.0075],
        "inverse_task.admm_config.pdhg.sigma_dual": [1200],
    },
}

for stage in cfg["stages"]:
    name = stage["name"]
    if name in stage_values:
        stage["parameters"] = {
            key: {"values": values}
            for key, values in stage_values[name].items()
        }

with custom_path.open("w", encoding="utf-8") as handle:
    yaml.safe_dump(cfg, handle, sort_keys=False)

print(f"Wrote {custom_path}")
print(custom_path.read_text(encoding='utf-8'))


In [ ]:
#@title Dry Run The Selected Stage
import os
import shlex
import subprocess
import sys

repo_dir = Path(REPO_DIR)
cmd = [
    sys.executable,
    "tuning/run_pdhg_tuning.py",
    "--config",
    CUSTOM_TUNING_CONFIG,
    "--stage",
    RUN_STAGE,
    "--dry-run",
]
if MAX_RUNS > 0:
    cmd.extend(["--max-runs", str(MAX_RUNS)])

print(" ".join(shlex.quote(part) for part in cmd))
subprocess.run(cmd, cwd=repo_dir, check=True)


In [ ]:
#@title Launch The Selected Stage
import os
import shlex
import subprocess
import sys

repo_dir = Path(REPO_DIR)
cmd = [
    sys.executable,
    "tuning/run_pdhg_tuning.py",
    "--config",
    CUSTOM_TUNING_CONFIG,
    "--stage",
    RUN_STAGE,
]
if MAX_RUNS > 0:
    cmd.extend(["--max-runs", str(MAX_RUNS)])

print(" ".join(shlex.quote(part) for part in cmd))
subprocess.run(cmd, cwd=repo_dir, check=True)


In [ ]:
#@title Show The Latest Leaderboard
from pathlib import Path
import pandas as pd

repo_dir = Path(REPO_DIR)
study_root = repo_dir / "tuning_runs" / "pdhg_phase_retrieval_colab"
if not study_root.exists():
    raise FileNotFoundError(f"No study root found at {study_root}")

latest_study = sorted(study_root.iterdir())[-1]
leaderboard = latest_study / "leaderboard.csv"
print(f"Latest study: {latest_study}")
df = pd.read_csv(leaderboard)
display(df)


In [ ]:
#@title Copy The Latest Study Back To Google Drive
from pathlib import Path
import shutil

repo_dir = Path(REPO_DIR)
study_root = repo_dir / "tuning_runs" / "pdhg_phase_retrieval_colab"
latest_study = sorted(study_root.iterdir())[-1]

export_root = Path(DRIVE_EXPORT_DIR)
export_root.mkdir(parents=True, exist_ok=True)
target = export_root / latest_study.name
if target.exists():
    shutil.rmtree(target)
shutil.copytree(latest_study, target)

print(f"Copied study summary to {target}")
print("Raw run artifacts remain in the repo's results/ and outputs/ folders inside the Colab VM unless you copy those separately.")


## 3. Next Iteration

After Stage A finishes:
- inspect the leaderboard
- update the values in `stage_values`
- change `RUN_STAGE`
- rerun the config-writing cell
- then rerun the dry-run and launch cells
